# Exploratory data analysis

Generation is the target (National Power Portal); the organiser CSV supplies only population and GSDP. This notebook shows the national NPP series, the NPP three-way consistency check and the CSV quirks.

In [1]:
import sys; sys.path.insert(0,'..')
import pandas as pd
from src.config import OUT, PROC
n=pd.read_csv(PROC/'national_series_fy.csv',index_col=0)
n[['Total','Thermal','Coal','Gas','Hydro','Nuclear','RES','gen_pc_kwh','gdp_pc']].tail(10).round(1)

,Total,Thermal,Coal,Gas,Hydro,Nuclear,RES,gen_pc_kwh,gdp_pc
FY,,,,,,,,,
2015,1094.4,877.9,NaN,NaN,129.2,36.1,51.1,870.0,82041.3
2016,1166.1,943.0,NaN,NaN,121.4,37.4,64.3,915.3,87967.3
2017,1232.1,993.5,NaN,NaN,122.4,37.9,78.3,955.1,94945.0
2018,1295.0,1037.1,NaN,NaN,126.1,38.3,93.4,992.9,100437.8
2019,1342.9,1072.3,1022.4,49.9,134.9,37.8,97.9,1018.4,105697.7
2020,1350.1,1042.7,994.3,48.4,155.8,46.5,105.1,1012.8,110058.1
2021,1337.8,1032.5,981.6,50.9,150.3,43.0,111.9,992.9,105753.3
2022,1442.2,1114.7,1078.7,36.0,151.6,47.1,128.8,1059.2,110058.2
2023,1586.7,1206.2,1182.3,23.9,162.1,45.9,172.5,1154.7,113627.7


In [2]:
c=pd.read_csv(OUT/'npp_consistency_check.csv')
print('max |diff| % (excl. Bhutan):', c[c.item!='Bhutan import'].diff_pct.abs().max().round(3))
c.round(2).head(12)

max |diff| % (excl. Bhutan): 1.301


,FY,item,a_source,a,b_source,b,diff_pct
0,2019,Thermal,NPP history endpoint,1072.31,NPP 18-col sub-report 2 (region-wise),1072.22,0.01
1,2019,Hydro,NPP history endpoint,134.89,NPP 18-col sub-report 2 (region-wise),134.89,0.00
2,2019,Nuclear,NPP history endpoint,37.81,NPP 18-col sub-report 2 (region-wise),37.81,0.00
3,2020,Thermal,NPP history endpoint,1042.75,NPP 18-col sub-report 2 (region-wise),1042.75,0.00
4,2020,Hydro,NPP history endpoint,155.77,NPP 18-col sub-report 2 (region-wise),155.77,-0.00
5,2020,Nuclear,NPP history endpoint,46.47,NPP 18-col sub-report 2 (region-wise),46.47,0.00
6,2021,Thermal,NPP history endpoint,1032.51,NPP 18-col sub-report 2 (region-wise),1032.51,0.00
7,2021,Hydro,NPP history endpoint,150.30,NPP 18-col sub-report 2 (region-wise),150.30,0.00
8,2021,Nuclear,NPP history endpoint,43.03,NPP 18-col sub-report 2 (region-wise),43.03,0.00
9,2022,Thermal,NPP history endpoint,1114.71,NPP 18-col sub-report 2 (region-wise),1114.71,-0.00


In [3]:
print(open(OUT/'data_quality_report.md').read()[:3000])

# Data-quality report

Generated by `src/clean.py`. Datasets: National Power Portal (npp.gov.in) and the organiser CSV only; no assumed values.

- Organiser CSV: 635 rows, 34 states/UTs, calendar years 2006-2024 (treated as FY2007-FY2025, i.e. CSV year Y = fiscal year Y-(Y+1)).
- Columns ['Installed - Bio-Mass Power', 'Installed - Small Hydro Power', 'Installed - Solar Power', 'Installed - Wind Power', 'Installed - Waste to Energy'] are INSTALLED CAPACITY (MW) -> never used for the level of generation or any comparison; only their composition (shares) apportions NPP's RES total among solar / wind / other and regions.
- 2024 capacity rows identical to 2023 (carry-forward): 19 of 34 -> 2024 rows flagged PROVISIONAL.
- CSV 'Zone' differs from the official CEA 5-region grid mapping for: Andaman And Nicobar Islands (South->ER), Chhattisgarh (Central->WR), Madhya Pradesh (Central->WR), Rajasthan (West->NR), Sikkim (North-East->ER). 'Central' is not a grid region. Remapped to NR/WR/SR/ER/NER.

In [4]:
pd.read_csv(PROC/'mix_region_base.csv',index_col=0).round(1)

,Coal,Gas,Nuclear,Hydro,Solar,Wind,Other RE
region,,,,,,,
NR,273.5,7.3,9.9,73.1,23.7,5.5,6.1
WR,520.6,12.2,15.7,16.6,48.2,22.1,4.8
SR,246.1,2.1,22.3,20.3,44.9,23.8,7.0
ER,250.0,0.0,0.0,17.8,1.4,0.1,1.0
NER,5.1,9.7,0.0,6.3,0.3,0.0,0.6
